# Phase 2: Chemical Space Analysis

Goals:
- Tanimoto similarity on Morgan fingerprints
- Bemis-Murcko scaffold analysis
- chemical descriptor distributions
- source–target chemical-space distance

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import pickle
import json
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Scaffolds
from rdkit.DataStructs import TanimotoSimilarity
from scipy.spatial.distance import pdist, squareform
from pathlib import Path

# Path settings
ROOT = r'..'
BENCHMARK_DIR = os.path.join(ROOT, 'benchmark')
RESULTS_DIR = os.path.join(BENCHMARK_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print("="*100)
print("PHASE 2: Chemical Space Analysis")
print("="*100)
print(f"Results: {RESULTS_DIR}\n")
np.random.seed(42)  # reproducible sampling

PHASE 2: Chemical Space Analysis
Results: ..\benchmark\results



In [2]:
# Load Phase 1 results
print("Loading Phase 1 results...\n")

with open(os.path.join(RESULTS_DIR, 'preprocessed_data.pkl'), 'rb') as f:
    phase1_data = pickle.load(f)

source_data = phase1_data['source_data']
target_data = phase1_data['target_data']
PROPERTIES = phase1_data['properties']
FOLD = phase1_data['fold']

print(f"Properties: {PROPERTIES}")
print(f"Fold: {FOLD}")
for prop in PROPERTIES:
    print(f"  {prop}: Source {len(source_data[prop])}, Target {len(target_data[prop])}")
print()

Loading Phase 1 results...

Properties: ['clearance', 'half_life', 'fu']
Fold: 0
  clearance: Source 14387, Target 1027
  half_life: Source 9204, Target 1164
  fu: Source 3783, Target 353



In [3]:
def get_morgan_fingerprint(smiles, radius=2, nbits=2048):
    """Generate Morgan fingerprints"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)
        else:
            return None
    except:
        return None

def get_bemis_murcko_scaffold(smiles):
    """Extract Bemis-Murcko scaffolds"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            scaffold = Scaffolds.MurckoScaffold.GetScaffoldForMol(mol)
            return Chem.MolToSmiles(scaffold)
        else:
            return None
    except:
        return None

def get_descriptors(smiles):
    """Compute molecular descriptors"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return {
                'mw': Descriptors.MolWt(mol),
                'logp': Descriptors.MolLogP(mol),
                'n_rings': Descriptors.RingCount(mol),
                'n_rotatable': Descriptors.NumRotatableBonds(mol),
                'hbd': Descriptors.NumHDonors(mol),
                'hba': Descriptors.NumHAcceptors(mol),
                'tpsa': Descriptors.TPSA(mol),
            }
        else:
            return None
    except:
        return None

print("✓ Chemical analysis functions defined")

✓ Chemical analysis functions defined


In [4]:
# Compute chemical-space features
chemical_space_results = {}

for prop in PROPERTIES:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    chemical_space_results[prop] = {}
    
    for domain_type, data_dict in [('source', source_data), ('target', target_data)]:
        df = data_dict[prop].copy()
        print(f"\n  {domain_type.upper()} ({len(df)} molecules):")
        
        # Morgan fingerprints
        print(f"    Computing Morgan fingerprints...")
        fps = []
        valid_smiles = []
        for idx, row in df.iterrows():
            fp = get_morgan_fingerprint(row['smiles'])
            if fp is not None:
                fps.append(fp)
                valid_smiles.append(row['smiles'])
        
        print(f"      Valid: {len(fps)}/{len(df)}")
        
        # Bemis-Murcko scaffolds
        print(f"    Computing Bemis-Murcko scaffolds...")
        scaffolds = []
        for smiles in valid_smiles:
            scaffold = get_bemis_murcko_scaffold(smiles)
            if scaffold is not None:
                scaffolds.append(scaffold)
            else:
                scaffolds.append(None)
        
        unique_scaffolds = len(set([s for s in scaffolds if s is not None]))
        print(f"      Unique scaffolds: {unique_scaffolds}")
        
        # Descriptors
        print(f"    Computing descriptors...")
        desc_list = []
        for smiles in valid_smiles:
            desc = get_descriptors(smiles)
            if desc is not None:
                desc_list.append(desc)
        
        desc_df = pd.DataFrame(desc_list)
        print(f"      MW: {desc_df['mw'].mean():.1f} ± {desc_df['mw'].std():.1f}")
        print(f"      LogP: {desc_df['logp'].mean():.2f} ± {desc_df['logp'].std():.2f}")
        print(f"      TPSA: {desc_df['tpsa'].mean():.1f} ± {desc_df['tpsa'].std():.1f}")
        
        # Store fingerprints and metadata
        chemical_space_results[prop][domain_type] = {
            'n_valid': int(len(fps)),
            'n_total': int(len(df)),
            'unique_scaffolds': int(unique_scaffolds),
            'fps': fps,
            'smiles': valid_smiles,
            'scaffolds': scaffolds[:len(valid_smiles)],
            'descriptors': desc_df.to_dict('list'),
        }

print("\n✓ Chemical space features computed")


Property: CLEARANCE

  SOURCE (14387 molecules):
    Computing Morgan fingerprints...
      Valid: 14387/14387
    Computing Bemis-Murcko scaffolds...
      Unique scaffolds: 0
    Computing descriptors...
      MW: 415.3 ± 116.8
      LogP: 3.61 ± 1.39
      TPSA: 79.0 ± 32.9

  TARGET (1027 molecules):
    Computing Morgan fingerprints...
      Valid: 1027/1027
    Computing Bemis-Murcko scaffolds...
      Unique scaffolds: 0
    Computing descriptors...
      MW: 406.2 ± 184.4
      LogP: 0.21 ± 3.19
      TPSA: 108.9 ± 69.2

Property: HALF_LIFE

  SOURCE (9204 molecules):
    Computing Morgan fingerprints...
      Valid: 9204/9204
    Computing Bemis-Murcko scaffolds...
      Unique scaffolds: 0
    Computing descriptors...
      MW: 453.1 ± 127.8
      LogP: 4.18 ± 1.48
      TPSA: 87.1 ± 39.5

  TARGET (1164 molecules):
    Computing Morgan fingerprints...
      Valid: 1164/1164
    Computing Bemis-Murcko scaffolds...
      Unique scaffolds: 0
    Computing descriptors...
      

In [5]:
# Intra-domain similarity (sampled to save memory)
print(f"\n{'='*80}")
print("INTRA-DOMAIN SIMILARITY (Tanimoto - Sampled)")
print(f"{'='*80}\n")

intra_similarity = {}

for prop in PROPERTIES:
    print(f"\n{prop.upper()}:")
    intra_similarity[prop] = {}
    
    for domain_type in ['source', 'target']:
        fps = chemical_space_results[prop][domain_type]['fps']
        
        if len(fps) < 2:
            print(f"  {domain_type}: Not enough molecules")
            continue
        
        # Sample fingerprints if too many (for memory efficiency)
        n_samples = min(len(fps), 1000)
        if len(fps) > n_samples:
            sample_idx = np.random.choice(len(fps), n_samples, replace=False)
            fps_sample = [fps[i] for i in sample_idx]
        else:
            fps_sample = fps
        
        # Compute pairwise similarities
        sim_list = []
        n_fps = len(fps_sample)
        
        for i in range(n_fps):
            for j in range(i+1, n_fps):
                sim = TanimotoSimilarity(fps_sample[i], fps_sample[j])
                sim_list.append(sim)
        
        if len(sim_list) > 0:
            mean_sim = np.mean(sim_list)
            std_sim = np.std(sim_list)
            
            intra_similarity[prop][domain_type] = {
                'mean': float(mean_sim),
                'std': float(std_sim),
                'min': float(np.min(sim_list)),
                'max': float(np.max(sim_list)),
                'n_pairs': int(len(sim_list)),
            }
            
            print(f"  {domain_type}: mean={mean_sim:.4f}±{std_sim:.4f} (n_pairs={len(sim_list)}, sampled={n_samples}/{len(fps)})")

print("\n✓ Intra-domain similarity computed")


INTRA-DOMAIN SIMILARITY (Tanimoto - Sampled)


CLEARANCE:
  source: mean=0.1201±0.0397 (n_pairs=499500, sampled=1000/14387)
  target: mean=0.0951±0.0458 (n_pairs=499500, sampled=1000/1027)

HALF_LIFE:
  source: mean=0.1170±0.0443 (n_pairs=499500, sampled=1000/9204)
  target: mean=0.0967±0.0454 (n_pairs=499500, sampled=1000/1164)

FU:
  source: mean=0.1247±0.0458 (n_pairs=499500, sampled=1000/3783)
  target: mean=0.0950±0.0420 (n_pairs=62128, sampled=353/353)

✓ Intra-domain similarity computed


In [6]:
# Inter-domain (source-target) similarity
print(f"\n{'='*80}")
print("INTER-DOMAIN SIMILARITY (Source vs Target - Sampled)")
print(f"{'='*80}\n")

inter_similarity = {}

for prop in PROPERTIES:
    print(f"\n{prop.upper()}:")
    
    fps_src = chemical_space_results[prop]['source']['fps']
    fps_tgt = chemical_space_results[prop]['target']['fps']
    
    if len(fps_src) == 0 or len(fps_tgt) == 0:
        print(f"  No data")
        inter_similarity[prop] = {}
        continue
    
    # Sample to reduce computation
    n_src_sample = min(len(fps_src), 500)
    n_tgt_sample = min(len(fps_tgt), 500)
    
    idx_src = np.random.choice(len(fps_src), n_src_sample, replace=False)
    idx_tgt = np.random.choice(len(fps_tgt), n_tgt_sample, replace=False)
    
    fps_src_sample = [fps_src[i] for i in idx_src]
    fps_tgt_sample = [fps_tgt[i] for i in idx_tgt]
    
    # Cross-domain similarities (vectorized to save memory)
    sim_list = []
    batch_size = 100
    
    for batch_start in range(0, len(fps_src_sample), batch_size):
        batch_end = min(batch_start + batch_size, len(fps_src_sample))
        
        for fp_src in fps_src_sample[batch_start:batch_end]:
            for fp_tgt in fps_tgt_sample:
                sim = TanimotoSimilarity(fp_src, fp_tgt)
                sim_list.append(sim)
    
    if len(sim_list) > 0:
        mean_sim = np.mean(sim_list)
        std_sim = np.std(sim_list)
        
        inter_similarity[prop] = {
            'mean': float(mean_sim),
            'std': float(std_sim),
            'min': float(np.min(sim_list)),
            'max': float(np.max(sim_list)),
            'n_comparisons': int(len(sim_list)),
        }
        
        print(f"  Source sample: {n_src_sample}/{len(fps_src)}")
        print(f"  Target sample: {n_tgt_sample}/{len(fps_tgt)}")
        print(f"  Mean similarity: {mean_sim:.4f}±{std_sim:.4f} (n={len(sim_list)})")

print("\n✓ Inter-domain similarity computed")


INTER-DOMAIN SIMILARITY (Source vs Target - Sampled)


CLEARANCE:
  Source sample: 500/14387
  Target sample: 500/1027
  Mean similarity: 0.0947±0.0344 (n=250000)

HALF_LIFE:
  Source sample: 500/9204
  Target sample: 500/1164
  Mean similarity: 0.0944±0.0345 (n=250000)

FU:
  Source sample: 500/3783
  Target sample: 353/353
  Mean similarity: 0.0966±0.0336 (n=176500)

✓ Inter-domain similarity computed


In [7]:
# Compare descriptor distributions
print(f"\n{'='*80}")
print("DESCRIPTOR DISTRIBUTION COMPARISON")
print(f"{'='*80}\n")

descriptor_comparison = {}
descriptor_names = ['mw', 'logp', 'n_rings', 'n_rotatable', 'hbd', 'hba', 'tpsa']

for prop in PROPERTIES:
    print(f"\n{prop.upper()}:")
    descriptor_comparison[prop] = {}
    
    desc_src = chemical_space_results[prop]['source']['descriptors']
    desc_tgt = chemical_space_results[prop]['target']['descriptors']
    
    for desc_name in descriptor_names:
        if desc_name in desc_src and desc_name in desc_tgt:
            src_vals = np.array(desc_src[desc_name])
            tgt_vals = np.array(desc_tgt[desc_name])
            
            # Wasserstein distance
            from scipy.stats import wasserstein_distance
            wd = float(wasserstein_distance(src_vals, tgt_vals))
            
            # KS test
            from scipy.stats import ks_2samp
            ks_stat, ks_pval = ks_2samp(src_vals, tgt_vals)
            
            descriptor_comparison[prop][desc_name] = {
                'source_mean': float(np.mean(src_vals)),
                'source_std': float(np.std(src_vals)),
                'target_mean': float(np.mean(tgt_vals)),
                'target_std': float(np.std(tgt_vals)),
                'wasserstein_distance': wd,
                'ks_stat': float(ks_stat),
                'ks_pval': float(ks_pval),
            }
            
            print(f"  {desc_name:12s}: src={np.mean(src_vals):7.2f}±{np.std(src_vals):5.2f} vs tgt={np.mean(tgt_vals):7.2f}±{np.std(tgt_vals):5.2f} (WD={wd:.3f})")

print("\n✓ Descriptor distributions compared")


DESCRIPTOR DISTRIBUTION COMPARISON


CLEARANCE:
  mw          : src= 415.34±116.75 vs tgt= 406.22±184.36 (WD=57.985)
  logp        : src=   3.61± 1.39 vs tgt=   0.21± 3.19 (WD=3.433)
  n_rings     : src=   3.98± 1.30 vs tgt=   3.25± 1.92 (WD=0.872)
  n_rotatable : src=   5.27± 2.86 vs tgt=   5.76± 4.19 (WD=0.976)
  hbd         : src=   1.36± 1.12 vs tgt=   2.54± 2.38 (WD=1.173)
  hba         : src=   5.41± 2.18 vs tgt=   5.82± 3.73 (WD=1.093)
  tpsa        : src=  78.99±32.91 vs tgt= 108.86±69.14 (WD=31.841)

HALF_LIFE:
  mw          : src= 453.08±127.79 vs tgt= 399.15±177.50 (WD=72.397)
  logp        : src=   4.18± 1.48 vs tgt=   0.37± 3.09 (WD=3.819)
  n_rings     : src=   4.17± 1.36 vs tgt=   3.21± 1.84 (WD=1.036)
  n_rotatable : src=   6.06± 3.38 vs tgt=   5.70± 4.08 (WD=0.740)
  hbd         : src=   1.68± 1.17 vs tgt=   2.43± 2.29 (WD=0.747)
  hba         : src=   5.68± 2.57 vs tgt=   5.59± 3.64 (WD=0.851)
  tpsa        : src=  87.13±39.52 vs tgt= 104.25±67.51 (WD=19.866)

FU:
  

In [8]:
# Compute scaffold overlap
print(f"\n{'='*80}")
print("SCAFFOLD OVERLAP ANALYSIS")
print(f"{'='*80}\n")

scaffold_overlap = {}

for prop in PROPERTIES:
    print(f"\n{prop.upper()}:")
    
    scaffolds_src = set([s for s in chemical_space_results[prop]['source']['scaffolds'] if s is not None])
    scaffolds_tgt = set([s for s in chemical_space_results[prop]['target']['scaffolds'] if s is not None])
    
    overlap = scaffolds_src & scaffolds_tgt
    
    tanimoto_overlap = len(overlap) / len(scaffolds_src | scaffolds_tgt) if len(scaffolds_src | scaffolds_tgt) > 0 else 0
    
    print(f"  Source unique scaffolds: {len(scaffolds_src)}")
    print(f"  Target unique scaffolds: {len(scaffolds_tgt)}")
    print(f"  Overlap: {len(overlap)}")
    print(f"  Jaccard index: {tanimoto_overlap:.4f}")
    
    scaffold_overlap[prop] = {
        'source_unique': int(len(scaffolds_src)),
        'target_unique': int(len(scaffolds_tgt)),
        'overlap': int(len(overlap)),
        'jaccard_index': float(tanimoto_overlap),
    }

print("\n✓ Scaffold overlap computed")


SCAFFOLD OVERLAP ANALYSIS


CLEARANCE:
  Source unique scaffolds: 0
  Target unique scaffolds: 0
  Overlap: 0
  Jaccard index: 0.0000

HALF_LIFE:
  Source unique scaffolds: 0
  Target unique scaffolds: 0
  Overlap: 0
  Jaccard index: 0.0000

FU:
  Source unique scaffolds: 0
  Target unique scaffolds: 0
  Overlap: 0
  Jaccard index: 0.0000

✓ Scaffold overlap computed


In [9]:
# Summary Report
print(f"\n{'='*100}")
print("CHEMICAL SPACE SUMMARY")
print(f"{'='*100}\n")

summary_data = []

for prop in PROPERTIES:
    row = {
        'Property': prop,
        'Source_Intra_Sim': f"{intra_similarity[prop]['source'].get('mean', 0):.4f}",
        'Target_Intra_Sim': f"{intra_similarity[prop]['target'].get('mean', 0):.4f}",
        'CrossDomain_Sim': f"{inter_similarity[prop].get('mean', 0):.4f}",
        'Scaffold_Jaccard': f"{scaffold_overlap[prop]['jaccard_index']:.4f}",
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print()


CHEMICAL SPACE SUMMARY

 Property Source_Intra_Sim Target_Intra_Sim CrossDomain_Sim Scaffold_Jaccard
clearance           0.1201           0.0951          0.0947           0.0000
half_life           0.1170           0.0967          0.0944           0.0000
       fu           0.1247           0.0950          0.0966           0.0000



In [10]:
# Save results
def convert_to_python_types(obj):
    """Convert numpy types to native Python types for JSON serialization"""
    if isinstance(obj, dict):
        return {k: convert_to_python_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_python_types(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj) if isinstance(obj, np.floating) else int(obj)
    return obj

# Remove non-serializable fingerprints before saving
chemical_space_results_json = {}
for prop in PROPERTIES:
    chemical_space_results_json[prop] = {}
    for domain_type in ['source', 'target']:
        data = chemical_space_results[prop][domain_type].copy()
        # Remove fingerprints (not JSON serializable)
        data.pop('fps', None)
        chemical_space_results_json[prop][domain_type] = data

with open(os.path.join(RESULTS_DIR, 'phase_02_intra_similarity.json'), 'w') as f:
    json.dump(convert_to_python_types(intra_similarity), f, indent=2)

with open(os.path.join(RESULTS_DIR, 'phase_02_inter_similarity.json'), 'w') as f:
    json.dump(convert_to_python_types(inter_similarity), f, indent=2)

with open(os.path.join(RESULTS_DIR, 'phase_02_descriptors.json'), 'w') as f:
    json.dump(convert_to_python_types(descriptor_comparison), f, indent=2)

with open(os.path.join(RESULTS_DIR, 'phase_02_scaffolds.json'), 'w') as f:
    json.dump(convert_to_python_types(scaffold_overlap), f, indent=2)

# Save preprocessed chemical data for Phase 3
with open(os.path.join(RESULTS_DIR, 'chemical_space_data.pkl'), 'wb') as f:
    pickle.dump(chemical_space_results, f)

print("✓ Results saved to benchmark/results/")
print(f"  - phase_02_intra_similarity.json")
print(f"  - phase_02_inter_similarity.json")
print(f"  - phase_02_descriptors.json")
print(f"  - phase_02_scaffolds.json")
print(f"  - chemical_space_data.pkl")
print("\nPhase 2 Complete! Ready for Phase 3: Representation Gap Analysis")

✓ Results saved to benchmark/results/
  - phase_02_intra_similarity.json
  - phase_02_inter_similarity.json
  - phase_02_descriptors.json
  - phase_02_scaffolds.json
  - chemical_space_data.pkl

Phase 2 Complete! Ready for Phase 3: Representation Gap Analysis
